# Simple harmonic oscillator

$$H(q, p) = \frac{p^2}{2m} + \frac{k q^2}{2}$$

This notebook derives Hamilton's equations with SymPy, integrates them with SciPy, and writes what the paper needs into `manuscript/generated/` and `manuscript/figures/`.

**Evidence:** the equations of motion are *computed* here; energy conservation is *proven* in Lean (Physlib, see `lean/Hamiltonian/Oscillator.lean`); the drift and the plots are *numerical*.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp

# The project root is the nearest folder containing manuscript/.
ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "manuscript").is_dir())
GENERATED = ROOT / "manuscript" / "generated"
FIGURES = ROOT / "manuscript" / "figures"
GENERATED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)


def latex_sci(x):
    """Format a number for LaTeX, e.g. 3.2e-11 -> $3.2 \\times 10^{-11}$."""
    mantissa, exponent = f"{x:.1e}".split("e")
    return f"${mantissa} \\times 10^{{{int(exponent)}}}$"

## Hamilton's equations

In [ ]:
q, p = sp.symbols("q p", real=True)
m, k = sp.symbols("m k", positive=True)

H = p**2 / (2 * m) + k * q**2 / 2
q_dot = sp.diff(H, p)
p_dot = -sp.diff(H, q)

sp.Eq(sp.Symbol(r"\dot q"), q_dot), sp.Eq(sp.Symbol(r"\dot p"), p_dot)

Along the flow, $dH/dt = \partial_q H\,\dot q + \partial_p H\,\dot p$, which vanishes identically:

In [ ]:
dH_dt = sp.simplify(sp.diff(H, q) * q_dot + sp.diff(H, p) * p_dot)
assert dH_dt == 0
dH_dt

In [ ]:
eom_tex = (
    "\\begin{align}\n"
    f"  \\dot q &= {sp.latex(q_dot)}, \\\\\n"
    f"  \\dot p &= {sp.latex(p_dot)}.\n"
    "\\end{align}\n"
)
(GENERATED / "oscillator_eom.tex").write_text(eom_tex, encoding="utf-8")
print(eom_tex)

## Numerical trajectories

In [ ]:
values = {m: 1.0, k: 1.0}
rhs = sp.lambdify((q, p), [q_dot.subs(values), p_dot.subs(values)], "numpy")
energy = sp.lambdify((q, p), H.subs(values), "numpy")

t_end = 4 * np.pi
t_eval = np.linspace(0, t_end, 2000)
trajectories = [
    solve_ivp(lambda t, y: rhs(*y), (0, t_end), [q0, 0.0], t_eval=t_eval, rtol=1e-10, atol=1e-12)
    for q0 in [0.5, 1.0, 1.5, 2.0]
]

drift = max(
    np.max(np.abs(energy(sol.y[0], sol.y[1]) - energy(sol.y[0][0], sol.y[1][0])))
    for sol in trajectories
)
(GENERATED / "oscillator_drift.tex").write_text(latex_sci(drift), encoding="utf-8")
print(f"Largest energy drift over {len(trajectories)} trajectories: {drift:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for sol in trajectories:
    ax.plot(sol.y[0], sol.y[1], lw=1.2, label=f"$q_0 = {sol.y[0][0]:.1f}$")
ax.set_xlabel("$q$")
ax.set_ylabel("$p$")
ax.set_aspect("equal")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "oscillator_phase.pdf")
plt.show()

An interactive version of the same plot (hover to read values):

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for sol in trajectories:
    fig.add_trace(go.Scatter(x=sol.y[0], y=sol.y[1], mode="lines", name=f"q0 = {sol.y[0][0]:.1f}"))
fig.update_layout(xaxis_title="q", yaxis_title="p", yaxis_scaleanchor="x", width=520, height=480)
fig.show()